# Assignment #03

by Patrick Donnelly & Burke Havranek

EECE 5644: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

### **1. Company Background**

WheelsBazaar is an online used-car marketplace operating across India (similar to Cars24, CarDekho, or
Spinny). Sellers list their vehicles on our platform, and buyers browse, compare, and purchase. We currently
handle roughly 5,000 listings per month across metros and tier-2 cities.

### **2. Business Problem**
Today, sellers set their own asking price, and our operations team manually reviews listings that "look off." This
causes three measurable problems:
1. **Overpriced listings sit unsold.** Cars priced more than ~15% above market value take 3x longer to sell,
clogging inventory and hurting buyer trust in the platform.
2. **Underpriced listings cost sellers money** — and generate complaints when sellers later discover they
sold below market. This damages retention.
3. **Manual review doesn't scale.** Our pricing analysts can review ~40 listings a day. At 5,000
listings/month we review less than 25% of inventory.
We also plan to launch an instant-buy program (WheelsBazaar Direct) where the company purchases cars
directly from sellers. For this, we need an auto

## Part 1: Data Retrieval and Sanitization

In [1]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

df = pd.read_csv("car_details_v4.csv")
df

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2054,Mahindra,XUV500 W8 [2015-2017],850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179 cc,138 bhp @ 3750 rpm,330 Nm @ 1600 rpm,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,Eon D-Lite +,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814 cc,55 bhp @ 5500 rpm,75 Nm @ 4000 rpm,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,Figo Duratec Petrol ZXI 1.2,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196 cc,70 bhp @ 6250 rpm,102 Nm @ 4000 rpm,FWD,3795.0,1680.0,1427.0,5.0,45.0
2057,BMW,5-Series 520d Luxury Line [2017-2019],4290000,2018,60474,Diesel,Automatic,Coimbatore,White,First,Individual,1995 cc,188 bhp @ 4000 rpm,400 Nm @ 1750 rpm,RWD,4936.0,1868.0,1479.0,5.0,65.0


Thus, we find 2059 entries with 20 individual fields. First, we examine any missing or malformed entries:

In [2]:
df.isnull().sum()

Make                    0
Model                   0
Price                   0
Year                    0
Kilometer               0
Fuel Type               0
Transmission            0
Location                0
Color                   0
Owner                   0
Seller Type             0
Engine                 80
Max Power              80
Max Torque             80
Drivetrain            136
Length                 64
Width                  64
Height                 64
Seating Capacity       64
Fuel Tank Capacity    113
dtype: int64

In [3]:
df.isna().sum()

Make                    0
Model                   0
Price                   0
Year                    0
Kilometer               0
Fuel Type               0
Transmission            0
Location                0
Color                   0
Owner                   0
Seller Type             0
Engine                 80
Max Power              80
Max Torque             80
Drivetrain            136
Length                 64
Width                  64
Height                 64
Seating Capacity       64
Fuel Tank Capacity    113
dtype: int64

We find several across several fields with missing entries. To determine whether they are correlated, let us examine first the most deficient field:

In [4]:
df[df["Drivetrain"].isnull()].head(10)

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
33,Honda,CR-V 2.4 AT,860000,2013,67000,Petrol,Automatic,Mumbai,Brown,First,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,Audi,A4 2.0 TDI (143 bhp),1250000,2012,50000,Diesel,Automatic,Mumbai,White,First,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,Mercedes-Benz,GLC 220 d Sport,3900000,2018,83400,Diesel,Automatic,Hyderabad,White,Second,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,Honda,Brio S MT,229000,2013,38175,Petrol,Manual,Kolkata,Blue,First,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117,Hyundai,Santro GL LPG,120000,2009,48500,LPG,Manual,Lucknow,Grey,Fourth,Individual,1086 cc,63@5500,89@3000,NaN,3565.0,1525.0,1590.0,5.0,35.0
138,Maruti Suzuki,Ritz VXI BS-IV,210000,2011,58888,Petrol,Manual,Delhi,Maroon,Second,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
147,Mercedes-Benz,GLC 220 d Sport,4400000,2018,47009,Diesel,Automatic,Pune,Blue,Second,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,MG,Hector Sharp 1.5 DCT Petrol [2019-2020],1729000,2020,27000,Petrol,Automatic,Panvel,Silver,First,Individual,1451 cc,141 bhp @ 5000 rpm,250 Nm @ 1600 rpm,NaN,4655.0,1835.0,1760.0,5.0,60.0
177,Maruti Suzuki,Wagon R VXi,350000,2013,45000,Petrol,Manual,Pune,White,First,Individual,998 cc,68@6200,90@3500,NaN,3539.0,1495.0,1700.0,5.0,35.0
192,Toyota,Innova 2.5 G4 7 STR,645000,2013,76000,Diesel,Manual,Kanpur,Grey,Second,Individual,2494 cc,102@5600,200@1400,NaN,4555.0,1770.0,1755.0,7.0,55.0


From this, we find that the spread of invalid entries tend to correlate quite strongly with each other, but not with any other fields in particular. We are unsure as to the relative importance of each field in determining price, however. Since these incomplete entries compose a rather small percentage of the total data set (~9%), and due to a lack of any defensible default value for the categorical value of `Drivetrain`, we have therefore chosen to drop incomplete entries.

In [5]:
df.dropna(inplace=True)
df

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,Ritz Vxi (ABS) BS-IV,245000,2014,79000,Petrol,Manual,Faridabad,White,Second,Individual,1197 cc,85 bhp @ 6000 rpm,113 Nm @ 4500 rpm,FWD,3775.0,1680.0,1620.0,5.0,43.0
2054,Mahindra,XUV500 W8 [2015-2017],850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179 cc,138 bhp @ 3750 rpm,330 Nm @ 1600 rpm,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,Eon D-Lite +,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814 cc,55 bhp @ 5500 rpm,75 Nm @ 4000 rpm,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,Figo Duratec Petrol ZXI 1.2,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196 cc,70 bhp @ 6250 rpm,102 Nm @ 4000 rpm,FWD,3795.0,1680.0,1427.0,5.0,45.0


Let us now consider the format of each entry for each field. For such entries as `Make` and `Model`, these fields are purely categorical, and cannot be meaningfully reduced further other than through encoding, as discussed below. However, for such fields as `Engine`, `Max Power`, and `Max Torque`, these fields carry continuous numerical information in addition to units, and may therefore uniformly sanitized as follows:

In [6]:
# Given a string, extracts the first floating-point number
get_first_number = lambda x: float(re.search(r'^\d+\.?\d*', x).group())

df["Engine"] = df["Engine"].apply(get_first_number)
df["Max Power"] = df["Max Power"].apply(get_first_number)
df["Max Torque"] = df["Max Torque"].apply(get_first_number)

df

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198.0,87.0,109.0000,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248.0,74.0,190.0000,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197.0,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197.0,82.0,113.0000,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393.0,148.0,343.0000,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,Ritz Vxi (ABS) BS-IV,245000,2014,79000,Petrol,Manual,Faridabad,White,Second,Individual,1197.0,85.0,113.0000,FWD,3775.0,1680.0,1620.0,5.0,43.0
2054,Mahindra,XUV500 W8 [2015-2017],850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179.0,138.0,330.0000,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,Eon D-Lite +,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814.0,55.0,75.0000,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,Figo Duratec Petrol ZXI 1.2,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196.0,70.0,102.0000,FWD,3795.0,1680.0,1427.0,5.0,45.0


Next, we deal with the categorical data, beginning with the number of unique entries:

In [7]:
df.nunique()

Make                   32
Model                 955
Price                 586
Year                   18
Kilometer             790
Fuel Type               7
Transmission            2
Location               75
Color                  16
Owner                   4
Seller Type             3
Engine                104
Max Power             156
Max Torque            135
Drivetrain              3
Length                234
Width                 163
Height                189
Seating Capacity        6
Fuel Tank Capacity     55
dtype: int64

As expected, the categories of `Fuel Type`, `Transmission`, `Owner`, `Seller Type`, and `Drivetrain` have few entries. For `Color`, `Make`, and especially `Location` and `Model` have significantly more unique entries, the latter most comprising almost half unique entries. It is at this point that we must determine what features are significant to the cost of a vehicle.

While the `Model` of a car certainly affects its price, the sheer number of unique entries relative to the number of total entries makes any model training unlikely to converge upon a pattern, having only on average about two vehicles per unique model. We must therefore drop the `Model` field so as to not introduce undue noise to the model training.

The `Make` of a car, however, is significant to its price, with luxury brands commanding on average higher prices than budget brands. As such, this field must be kept.

The `Location` and `Color` of a car are both interesting pieces of categorical data, as it is unclear whether they might affect the car's final price. For now, we keep them, but may later drop them during model tuning.

In [8]:
df.drop("Model", axis=1, inplace=True)
df

,Make,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198.0,87.0,109.0000,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248.0,74.0,190.0000,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197.0,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197.0,82.0,113.0000,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393.0,148.0,343.0000,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,245000,2014,79000,Petrol,Manual,Faridabad,White,Second,Individual,1197.0,85.0,113.0000,FWD,3775.0,1680.0,1620.0,5.0,43.0
2054,Mahindra,850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179.0,138.0,330.0000,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814.0,55.0,75.0000,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196.0,70.0,102.0000,FWD,3795.0,1680.0,1427.0,5.0,45.0


Let us finally verify the aforementioned categorical data:

In [9]:
print(df["Make"].unique(),end='\n\n')
print(df["Fuel Type"].unique(),end='\n\n')
print(df["Transmission"].unique(),end='\n\n')
print(df["Location"].unique(),end='\n\n')
print(df["Color"].unique(),end='\n\n')
print(df["Owner"].unique(),end='\n\n')
print(df["Seller Type"].unique(),end='\n\n')
print(df["Drivetrain"].unique(),end='\n\n')

['Honda' 'Maruti Suzuki' 'Hyundai' 'Toyota' 'BMW' 'Skoda' 'Nissan'
 'Renault' 'Tata' 'Volkswagen' 'Ford' 'Mercedes-Benz' 'Audi' 'Mahindra'
 'MG' 'Jeep' 'Porsche' 'Kia' 'Land Rover' 'Volvo' 'Maserati' 'Jaguar'
 'Isuzu' 'MINI' 'Ferrari' 'Mitsubishi' 'Datsun' 'Chevrolet' 'Ssangyong'
 'Fiat' 'Rolls-Royce' 'Lexus']

['Petrol' 'Diesel' 'CNG' 'CNG + CNG' 'LPG' 'Hybrid' 'Petrol + CNG']

['Manual' 'Automatic']

['Pune' 'Ludhiana' 'Lucknow' 'Mangalore' 'Mumbai' 'Coimbatore' 'Bangalore'
 'Delhi' 'Raipur' 'Kanpur' 'Patna' 'Vadodara' 'Hyderabad' 'Yamunanagar'
 'Gurgaon' 'Jaipur' 'Deoghar' 'Agra' 'Goa' 'Warangal' 'Jalandhar' 'Noida'
 'Ahmedabad' 'Mohali' 'Ghaziabad' 'Kolkata' 'Zirakpur' 'Nagpur' 'Thane'
 'Faridabad' 'Ranchi' 'Chandigarh' 'Amritsar' 'Chennai' 'Navi Mumbai'
 'Udupi' 'Jamshedpur' 'Aurangabad' 'Rudrapur' 'Nashik' 'Varanasi' 'Salem'
 'Dehradun' 'Valsad' 'Haldwani' 'Dharwad' 'Surat' 'Indore' 'Karnal'
 'Panchkula' 'Mysore' 'Rohtak' 'Ambala Cantt' 'Samastipur' 'Panvel'
 'Purnea' 'Bhubaneswa

Everything looks correct, with no discernable typos save for `FuelType`, in which there does not appear to be a distinction between `CNG` and `CNG + CNG`, hence the two entries will be consolidated:

In [10]:
df["Fuel Type"] = df["Fuel Type"].replace("CNG + CNG", "CNG")
df["Fuel Type"].unique()

array(['Petrol', 'Diesel', 'CNG', 'LPG', 'Hybrid', 'Petrol + CNG'],
      dtype=object)

Lastly, the `Year`, `Transmission`, and `Owner` fields may be made generally numerically rather than categorical, producing `Age`, `Manual`, and `Owners` fields, respectively. Dropping the categorical fields, we produce:

In [11]:
df["Age"] = 2026 - df["Year"]
df.drop("Year", axis=1, inplace=True)

df["Manual"] = df["Transmission"].map({"Automatic": 1, "Manual": 0})
df.drop("Transmission", axis=1, inplace=True)

df["Owners"] = df["Owner"].map({"UnRegistered Car": 0, "First": 1, "Second": 2, "Third": 3})
df.drop("Owner", axis=1, inplace=True)

df

,Make,Price,Kilometer,Fuel Type,Location,Color,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity,Age,Manual,Owners
0,Honda,505000,87150,Petrol,Pune,Grey,Corporate,1198.0,87.0,109.0000,FWD,3990.0,1680.0,1505.0,5.0,35.0,9,0,1
1,Maruti Suzuki,450000,75000,Diesel,Ludhiana,White,Individual,1248.0,74.0,190.0000,FWD,3995.0,1695.0,1555.0,5.0,42.0,12,0,2
2,Hyundai,220000,67000,Petrol,Lucknow,Maroon,Individual,1197.0,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0,15,0,1
3,Toyota,799000,37500,Petrol,Mangalore,Red,Individual,1197.0,82.0,113.0000,FWD,3995.0,1745.0,1510.0,5.0,37.0,7,0,1
4,Toyota,1950000,69000,Diesel,Mumbai,Grey,Individual,2393.0,148.0,343.0000,RWD,4735.0,1830.0,1795.0,7.0,55.0,8,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,245000,79000,Petrol,Faridabad,White,Individual,1197.0,85.0,113.0000,FWD,3775.0,1680.0,1620.0,5.0,43.0,12,0,2
2054,Mahindra,850000,90300,Diesel,Surat,White,Individual,2179.0,138.0,330.0000,FWD,4585.0,1890.0,1785.0,7.0,70.0,10,0,1
2055,Hyundai,275000,83000,Petrol,Ahmedabad,White,Individual,814.0,55.0,75.0000,FWD,3495.0,1550.0,1500.0,5.0,32.0,12,0,2
2056,Ford,240000,73000,Petrol,Thane,Silver,Individual,1196.0,70.0,102.0000,FWD,3795.0,1680.0,1427.0,5.0,45.0,13,0,1


Incoding needs to be done on the data for all sections in the data frame with text. Most likely the location does not help but by using a LASSO model it should go to 0 if it does not matter.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1874 entries, 0 to 2057
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Make                1874 non-null   object 
 1   Price               1874 non-null   int64  
 2   Kilometer           1874 non-null   int64  
 3   Fuel Type           1874 non-null   object 
 4   Location            1874 non-null   object 
 5   Color               1874 non-null   object 
 6   Seller Type         1874 non-null   object 
 7   Engine              1874 non-null   float64
 8   Max Power           1874 non-null   float64
 9   Max Torque          1874 non-null   float64
 10  Drivetrain          1874 non-null   object 
 11  Length              1874 non-null   float64
 12  Width               1874 non-null   float64
 13  Height              1874 non-null   float64
 14  Seating Capacity    1874 non-null   float64
 15  Fuel Tank Capacity  1874 non-null   float64
 16  Age        

In [ ]:
# ['Petrol' 'Diesel' 'CNG' 'Petrol + CNG' 'LPG' 'Hybrid']

df["Fuel_Type"] = df["Fuel Type"].map({"Petrol": 0, "Diesel": 1, "CNG": 2, "Petrol + CNG": 3, "LPG": 4, "Hybrid":5})
df.drop("Fuel Type", axis=1, inplace=True)

# Drivetrain ['FWD' 'RWD' 'AWD']

df["Drive_Train"] = df["Drivetrain"].map({"FWD": 0, "RWD": 1, "AWD": 2})
df.drop("Drivetrain", axis=1, inplace=True)

# Seller Type ['Corporate' 'Individual' 'Commercial Registration']

df["Seller_Type"] = df["Seller Type"].map({"Corporate": 0, "Individual": 1, "Commercial Registration": 2})
df.drop("Seller Type", axis=1, inplace=True)

# Color ['Grey' 'White' 'Maroon' 'Red' 'Blue' 'Orange' 'Silver' 'Brown' 'Black' 'Bronze' 'Gold' 'Beige' 'Green' 'Purple' 'Others' 'Yellow']

df["Colors"] = df["Color"].map({"Grey": 0, "White": 1, "Maroon": 2, "Red": 3, "Blue": 4, "Orange":5,"Silver":6,"Brown": 7, "Black": 8, "Bronze": 9, "Gold": 10, "Beige": 11, "Green":12,"Purple":13, "Others":14, "Yellow":15 })
df.drop("Color", axis=1, inplace=True)

# Make ['Honda' 'Maruti Suzuki' 'Hyundai' 'Toyota' 'BMW' 'Skoda' 'Nissan' 'Renault' 'Tata' 'Volkswagen' 'Ford' 'Mercedes-Benz' 'Audi' 'Mahindra' 'MG' 'Jeep' 'Porsche' 'Kia' 'Land Rover' 'Volvo' 'Maserati' 'Jaguar' 'Isuzu' 'MINI' 'Ferrari' 'Mitsubishi' 'Datsun' 'Chevrolet' 'Ssangyong' 'Fiat' 'Rolls-Royce' 'Lexus']

df["Makes"] = df["Make"].map({"Honda": 0, "Maruti Suzuki": 1, "Hyundai": 2, "Toyota": 3, "BMW": 4, "Skoda":5,"Nissan":6,"Renault": 7, "Tata": 8, "Volkswagen": 9, "Ford": 10, "Mercedes-Benz": 11, "Audi":12,"Mahindra":13, "MG":14, "Jeep":15, "Porsche":16, "Kia":17, "Land Rover":18, "Volvo":19, "Maserati":20,"Jaguar":21, "Isuzu":22, "MINI":23, "Ferrari" :24,"Mitsubishi":25,"Datsun":26, "Chevrolet":27, "Ssangyong":28,"Fiat":29,"Rolls-Royce":30,"Lexus":31})
df.drop("Make", axis=1, inplace=True)

# Location ['Pune' 'Ludhiana' 'Lucknow' 'Mangalore' 'Mumbai' 'Coimbatore' 'Bangalore' 'Delhi' 'Raipur' 'Kanpur' 'Patna' 'Vadodara' 'Hyderabad' 'Yamunanagar'
# 'Gurgaon' 'Jaipur' 'Deoghar' 'Agra' 'Goa' 'Warangal' 'Jalandhar' 'Noida' 'Ahmedabad' 'Mohali' 'Ghaziabad' 'Kolkata' 'Zirakpur' 'Nagpur' 'Thane'
# 'Faridabad' 'Ranchi' 'Chandigarh' 'Amritsar' 'Chennai' 'Navi Mumbai' 'Udupi' 'Jamshedpur' 'Aurangabad' 'Rudrapur' 'Nashik' 'Varanasi' 'Salem'
# 'Dehradun' 'Valsad' 'Haldwani' 'Dharwad' 'Surat' 'Indore' 'Karnal' 'Panchkula' 'Mysore' 'Rohtak' 'Ambala Cantt' 'Samastipur' 'Panvel'
# 'Purnea' 'Bhubaneswar' 'Kheda' 'Kollam' 'Meerut' 'Ernakulam' 'Kharar' 'Mirzapur' 'Bhopal' 'Gorakhpur' 'Guwahati' 'Allahabad' 'Muzaffurpur'
# 'Faizabad' 'Kota' 'Pimpri-Chinchwad' 'Dak. Kannada' 'Ranga Reddy' 'Bulandshahar' 'Roorkee']

# df["Locations"] = df["Location"].map({"Honda": 0, "Maruti Suzuki": 1, "Hyundai": 2, "Toyota": 3, "BMW": 4, "Skoda":5,"Nissan":6,"Renault": 7, "Tata": 8, "Volkswagen": 9, "Ford": 10, "Mercedes-Benz": 11, "Audi":12,"Mahindra":13, "MG":14, "Jeep":15, "Porsche":16, "Kia":17, "Land Rover":18, "Volvo":19, "Maserati":20,"Jaguar":21, "Isuzu":22, "MINI":23, "Ferrari" :24,"Mitsubishi":25,"Datsun":26, "Chevrolet":27, "Ssangyong":28,"Fiat":29,"Rolls-Royce":30,"Lexus":31})

df.drop("Location", axis=1, inplace=True) # will need to be changed at some point to include location


,Price,Kilometer,Location,Engine,Max Power,Max Torque,Length,Width,Height,Seating Capacity,Fuel Tank Capacity,Age,Manual,Owners,Fuel_Type,Drive_Train,Seller_Type,Colors,Makes
0,505000,87150,Pune,1198.0,87.0,109.0000,3990.0,1680.0,1505.0,5.0,35.0,9,0,1,0,0,0,0,0
1,450000,75000,Ludhiana,1248.0,74.0,190.0000,3995.0,1695.0,1555.0,5.0,42.0,12,0,2,1,0,1,1,1
2,220000,67000,Lucknow,1197.0,79.0,112.7619,3585.0,1595.0,1550.0,5.0,35.0,15,0,1,0,0,1,2,2
3,799000,37500,Mangalore,1197.0,82.0,113.0000,3995.0,1745.0,1510.0,5.0,37.0,7,0,1,0,0,1,3,3
4,1950000,69000,Mumbai,2393.0,148.0,343.0000,4735.0,1830.0,1795.0,7.0,55.0,8,0,1,1,1,1,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,245000,79000,Faridabad,1197.0,85.0,113.0000,3775.0,1680.0,1620.0,5.0,43.0,12,0,2,0,0,1,1,1
2054,850000,90300,Surat,2179.0,138.0,330.0000,4585.0,1890.0,1785.0,7.0,70.0,10,0,1,1,0,1,1,13
2055,275000,83000,Ahmedabad,814.0,55.0,75.0000,3495.0,1550.0,1500.0,5.0,32.0,12,0,2,0,0,1,1,2
2056,240000,73000,Thane,1196.0,70.0,102.0000,3795.0,1680.0,1427.0,5.0,45.0,13,0,1,0,0,1,6,10


Normalization needs to be done on elements that are not encoded.

In [ ]:
from sklearn.preprocessing import #Normalization most likley in "StandardScaler"

## Part 2: Model Creation

For the Model of choice we will be using LASSO. This model will be good for this perpuse do to the ability to discard elements that are not meaningful while we do not know that is needed for a good prediction or not.

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1874 entries, 0 to 2057
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Price               1874 non-null   int64  
 1   Kilometer           1874 non-null   int64  
 2   Location            1874 non-null   object 
 3   Engine              1874 non-null   float64
 4   Max Power           1874 non-null   float64
 5   Max Torque          1874 non-null   float64
 6   Length              1874 non-null   float64
 7   Width               1874 non-null   float64
 8   Height              1874 non-null   float64
 9   Seating Capacity    1874 non-null   float64
 10  Fuel Tank Capacity  1874 non-null   float64
 11  Age                 1874 non-null   int64  
 12  Manual              1874 non-null   int64  
 13  Owners              1874 non-null   int64  
 14  Fuel_Type           1874 non-null   int64  
 15  Drive_Train         1874 non-null   int64  
 16  Seller_Type

In [ ]:
from sklearn.model_selection import train_test_split

CONTRIBUTERS = ["Kilometer", "Engine", "Max Power", "Max Torque", "Length", "Width", \
            "Height", "Seating Capacity", "Fuel Tank Capacity", "Age", "Manual", "Owners", "Fuel_type", "Drive_Train", "Seller_Type", "Colors", "Makes"]
TARGET = "Price"
X = df[CONTRIBUTERS]
Y = df[TARGET]

X_train, X_test, Y_train, Y_test = \
    train_test_split(X, Y, test_size=0.2, random_state=0xDEADBEEF)

from sklearn.linear_model import Lasso

model_lasso_tenths = Lasso(alpha=0.1, random_state=0xDEADBEEF, max_iter=10000)
model_lasso_ones = Lasso(alpha=1, random_state=0xDEADBEEF, max_iter=10000)
model_lasso_hundreds = Lasso(alpha=0.01, random_state=0xDEADBEEF, max_iter=10000)


# some prescaling needs to be done

# y_test_pred_tenths = model_lasso_tenths.predict(X_test_scaled)
# y_test_pred_ones = model_lasso_ones.predict(X_test_scaled)
# y_test_pred_hundreds = model_lasso_hundreds.predict(X_test_scaled)

TypeError: 'DataFrame' object is not callable

## Part 3: Model Evaluation

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print ("y_test_pred_tenths Evaluation")
print("R^2 :", r2_score(Y_test, y_test_pred_tenths))
print("MSA:", mean_squared_error(Y_test, y_test_pred_tenths))

print ("y_test_pred_ones Evaluation")
print("R^2 :", r2_score(Y_test, y_test_pred_ones))
print("MSA:", mean_squared_error(Y_test, y_test_pred_ones))

print ("y_test_pred_hundreds Evaluation")
print("R^2 :", r2_score(Y_test, y_test_pred_hundreds))
print("MSA:", mean_squared_error(Y_test, y_test_pred_hundreds))